In [1]:
!pip install -q ultralytics opencv-python

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 32.9 MB/s eta 0:00:00


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import cv2
import numpy as np
from ultralytics import YOLO
from IPython.display import Video, display

# =========================================================
# CONFIG
# =========================================================
INPUT_VIDEO = "/content/drive/MyDrive/dance.mp4"   # change this
OUTPUT_VIDEO = "/content/output_skeleton.mp4"
MODEL_NAME = "yolo11n-pose.pt"   # you can also try yolo11s-pose.pt
PERSON_CONF = 0.25
KEYPOINT_CONF = 0.40
LINE_THICKNESS = 3
JOINT_RADIUS = 4
DRAW_WHITE_ON_BLACK = True
SMOOTHING_ALPHA = 0.35
ONLY_LARGEST_PERSON = True

# COCO 17-keypoint skeleton pairs
SKELETON = [
    (5, 6),                 # shoulders
    (5, 7), (7, 9),         # left arm
    (6, 8), (8, 10),        # right arm
    (5, 11), (6, 12),       # torso sides
    (11, 12),               # hips
    (11, 13), (13, 15),     # left leg
    (12, 14), (14, 16)      # right leg
]

HEAD_CONNECTIONS = [
    (0, 1), (0, 2), (1, 3), (2, 4)
]

DRAW_HEAD = True


# =========================================================
# HELPERS
# =========================================================
def choose_person_index(boxes_xyxy):
    if len(boxes_xyxy) == 0:
        return None
    areas = (boxes_xyxy[:, 2] - boxes_xyxy[:, 0]) * (boxes_xyxy[:, 3] - boxes_xyxy[:, 1])
    return int(np.argmax(areas))


def smooth_points(prev_pts, curr_pts, alpha=0.35):
    if prev_pts is None:
        return curr_pts

    smoothed = {}
    all_keys = set(prev_pts.keys()).union(curr_pts.keys())

    for k in all_keys:
        prev = prev_pts.get(k, None)
        curr = curr_pts.get(k, None)

        if curr is None and prev is None:
            smoothed[k] = None
        elif curr is None:
            smoothed[k] = prev
        elif prev is None:
            smoothed[k] = curr
        else:
            x = int((1 - alpha) * prev[0] + alpha * curr[0])
            y = int((1 - alpha) * prev[1] + alpha * curr[1])
            smoothed[k] = (x, y)

    return smoothed


def make_canvas(height, width, white_on_black=True):
    if white_on_black:
        return np.zeros((height, width, 3), dtype=np.uint8)
    return np.ones((height, width, 3), dtype=np.uint8) * 255


def draw_skeleton(canvas, points, skeleton_pairs, joint_radius=4, line_thickness=3):
    h, w = canvas.shape[:2]

    # choose line/joint color based on background
    mean_val = canvas.mean()
    color = (255, 255, 255) if mean_val < 127 else (0, 0, 0)

    # draw lines
    for a, b in skeleton_pairs:
        pa = points.get(a, None)
        pb = points.get(b, None)
        if pa is not None and pb is not None:
            cv2.line(canvas, pa, pb, color, line_thickness, lineType=cv2.LINE_AA)

    # draw joints
    for _, pt in points.items():
        if pt is not None:
            x, y = pt
            if 0 <= x < w and 0 <= y < h:
                cv2.circle(canvas, (x, y), joint_radius, color, -1, lineType=cv2.LINE_AA)

    return canvas


# =========================================================
# MAIN
# =========================================================
def main():
    print("Loading model...")
    model = YOLO(MODEL_NAME)

    cap = cv2.VideoCapture(INPUT_VIDEO)
    if not cap.isOpened():
        raise RuntimeError(f"Could not open video: {INPUT_VIDEO}")

    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = cap.get(cv2.CAP_PROP_FPS)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    if fps <= 0:
        fps = 25.0

    print(f"Video opened: {width}x{height}, fps={fps}, frames={total_frames}")

    writer = cv2.VideoWriter(
        OUTPUT_VIDEO,
        cv2.VideoWriter_fourcc(*"mp4v"),
        fps,
        (width, height)
    )

    prev_points = None
    frame_idx = 0

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        results = model.predict(
            source=frame,
            conf=PERSON_CONF,
            verbose=False
        )

        canvas = make_canvas(height, width, white_on_black=DRAW_WHITE_ON_BLACK)
        current_points = {}

        if len(results) > 0:
            result = results[0]

            if result.boxes is not None and result.keypoints is not None:
                boxes = result.boxes.xyxy.cpu().numpy() if result.boxes.xyxy is not None else np.empty((0, 4))
                kpts_xy = result.keypoints.xy.cpu().numpy() if result.keypoints.xy is not None else None
                kpts_conf = result.keypoints.conf.cpu().numpy() if result.keypoints.conf is not None else None

                if kpts_xy is not None and len(kpts_xy) > 0:
                    if ONLY_LARGEST_PERSON and len(boxes) > 1:
                        person_idx = choose_person_index(boxes)
                    else:
                        person_idx = 0

                    if person_idx is not None:
                        person_xy = kpts_xy[person_idx]
                        person_cf = (
                            kpts_conf[person_idx]
                            if kpts_conf is not None
                            else np.ones((person_xy.shape[0],), dtype=float)
                        )

                        for j in range(person_xy.shape[0]):
                            x, y = person_xy[j]
                            c = float(person_cf[j])

                            if c >= KEYPOINT_CONF:
                                current_points[j] = (int(x), int(y))
                            else:
                                current_points[j] = None

        current_points = smooth_points(prev_points, current_points, alpha=SMOOTHING_ALPHA)
        prev_points = current_points

        canvas = draw_skeleton(
            canvas,
            current_points,
            SKELETON,
            joint_radius=JOINT_RADIUS,
            line_thickness=LINE_THICKNESS
        )

        if DRAW_HEAD:
            canvas = draw_skeleton(
                canvas,
                current_points,
                HEAD_CONNECTIONS,
                joint_radius=JOINT_RADIUS,
                line_thickness=max(1, LINE_THICKNESS - 1)
            )

        writer.write(canvas)
        frame_idx += 1

        if frame_idx % 30 == 0:
            print(f"Processed {frame_idx}/{total_frames} frames")

    cap.release()
    writer.release()

    print(f"Done. Saved output to: {OUTPUT_VIDEO}")
    return OUTPUT_VIDEO


output_path = main()

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Loading model...
Video opened: 1280x720, fps=24.0, frames=1236
Processed 30/1236 frames
Processed 60/1236 frames
Processed 90/1236 frames
Processed 120/1236 frames
Processed 150/1236 frames
Processed 180/1236 frames
Processed 210/1236 frames
Processed 240/1236 frames
Processed 270/1236 frames
Processed 300/1236 frames
Processed 330/1236 frames
Processed 360/1236 frames
Processed 390/1236 frames
Processed 420/1236 frames
Processed 450/1236 frames
Processed 480/1236 frames
Processed 510/1236 frames
Processed 540/1236 frames
Processed 570/1236 frames
Processed 600/1236 frames
Processed 630/1236 frames
Processed 660/1236 frames
Processed 690/1236 frames
Processed 720/1236 frames
Proce

In [4]:
from google.colab.files import download
download(OUTPUT_VIDEO)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>